# PhillyStat360 — 04b: Model Validation (Python / Colab port)

Port of `code/04b_model_validation.Rmd`. Extends 04a with four validation
components:

1. **Spatial cross-validation** by ZIP code (`GroupKFold`) — prevents the
   spatial-autocorrelation leakage that inflates standard k-fold estimates.
2. **LOGO CV** (`LeaveOneGroupOut`) — holds out one entire ZIP at a time;
   the hardest generalization test for citywide deployment.
3. **Prediction confidence intervals** — for `RandomForestClassifier`,
   computed from the variance across individual decision trees (a Python
   analog to ranger's infinitesimal jackknife).
4. **Sanity checks** — feature importance, calibration curve, partial
   dependence, known-vacant scoring.

> **Production model is the ensemble (Logit + RF), `ensemble_prob` from
> 04a.** Spatial CV / LOGO / CIs in this notebook are computed on the RF
> half because (a) only RF has tree variance for CI estimation and (b)
> spatial CV on RF is conservative — Logit is a simpler model that
> generalizes more smoothly, so the ensemble's spatial AUC will be at
> least as good as RF alone. Calibration plots, known-vacant scoring,
> and the high-uncertainty filter all use **`ensemble_prob`** (production
> score) so the validation reflects what the dashboard actually shows.

> **Runtime warning.** Spatial CV refits 10 RF models, LOGO refits one per
> ZIP (~45). Both default to a 10% subsample (sample flags below) so the
> notebook completes in ~10 min. Set the flags to `False` for a final pass.


## 0. Setup

In [ ]:
# Mount Google Drive (run once per Colab session)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
from pathlib import Path

ROOT       = Path('/content/drive/MyDrive/PhillyStat_R/PhillyStat360')
RAW_PATH   = ROOT / 'rawdata'         # original CSVs / geojsons
DATA_PATH  = ROOT / 'data'            # R-side outputs (features_residential.csv, ovs_residential.csv)
PY_PATH    = ROOT / 'data_py'         # 04a Python outputs (predictions, models, calibrators)
OUT_PATH   = ROOT / 'data_py'         # we WRITE here too
GRAPH_PATH = ROOT / 'graphs' / 'python'

OUT_PATH.mkdir(parents=True, exist_ok=True)
GRAPH_PATH.mkdir(parents=True, exist_ok=True)

assert (PY_PATH / 'all_predictions_rf.csv').exists(), \
    f'04a outputs not found at {PY_PATH}. Run 04a_tidymodeling.ipynb first.'
print('All paths OK. PY_PATH =', PY_PATH)


In [ ]:
import json
import joblib
import warnings
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (roc_auc_score, average_precision_score,
                             roc_curve, brier_score_loss,
                             confusion_matrix, classification_report)

warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid', context='talk')
SEED = 42

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.model_selection import (GroupKFold, LeaveOneGroupOut,
                                     train_test_split)
from sklearn.calibration import calibration_curve
from sklearn.inspection import partial_dependence

TRAIN_CUTOFF = pd.Timestamp('2025-10-01')


## 1. Load data

In [ ]:
features_df = pd.read_csv(DATA_PATH / 'features_residential.csv', low_memory=False)
preds       = pd.read_csv(PY_PATH   / 'all_predictions_rf.csv',  low_memory=False)
preds['parcel_number'] = preds['parcel_number'].astype(str)
thresholds  = pd.read_csv(PY_PATH   / 'model_thresholds.csv')

# Load 04a's pre-fit logit and ensemble calibrator so we can score test parcels
# with the production ensemble without refitting.
logit_fit  = joblib.load(PY_PATH / 'model_logit_final.joblib')
calibrators = joblib.load(PY_PATH / 'calibrators.joblib')
cal_ens     = calibrators['ensemble']
print('Loaded 04a artifacts: model_logit_final.joblib + calibrators.joblib')

ens_thresh = float(thresholds.loc[thresholds['model'] == 'ensemble', 'threshold'].iloc[0])
print(f"features_residential: {len(features_df):,} rows  |  OVS=1: {features_df['ovs'].mean():.1%}")
print(f"predictions:          {len(preds):,} rows")
print(f"Ensemble Youden threshold from 04a: {ens_thresh:.4f} (calibrated)")

# Verify ensemble columns are present
required = ['ensemble_prob', 'ensemble_prob_raw', 'ensemble_flag', 'risk_score']
missing  = [c for c in required if c not in preds.columns]
assert not missing, (
    f"Missing ensemble columns {missing} in all_predictions_rf.csv. "
    f"Re-run 04a_tidymodeling.ipynb."
)


## 2. Rebuild model dataset (mirrors 04a)

Same `model_vars`, same imputation, same split. Only used to refit RF for the
spatial / LOGO CV folds.


In [ ]:
# Mirrors the 04a model_vars (post-2026-04-28 audit, with C&S history)
model_vars = [
    'n_violations_total', 'n_violations_recent',
    'n_violations_2yr',   'n_violations_3yr',   'n_violations_5yr',
    'n_distinct_codes',
    'viol_trend_3v5', 'viol_accel_2v3',
    'n_repeat_codes', 'resolution_rate',
    'has_fire_safety_code',
    'days_since_last_viol',
    'license_lapse_rate',
    'exterior_condition', 'building_age',
    'log_livable_area',   'is_poor_condition',
    'years_since_sale',
    'n_cs_total', 'cs_span_days', 'days_since_last_cs',
    'n_transfers_total', 'n_transfers_5yr', 'n_transfers_3yr',
    'n_deed_transfers',
    'had_sheriff_sale', 'sheriff_sale_recent', 'n_sheriff_sales',
    'log_price_change',
    'days_since_last_transfer',
    'nbr_ovs_rate_zip', 'nbr_ovs_rate_tract',
    'nbr_n_vacant_zip', 'nbr_n_vacant_tract',
]
print(f"Total features in model_vars: {len(model_vars)}")


In [ ]:
model_df = features_df.dropna(subset=['exterior_condition', 'building_age']).copy()

for col in ['days_since_last_viol', 'days_since_last_transfer']:
    if col in model_df.columns and model_df[col].isna().any():
        model_df[col] = model_df[col].fillna(model_df[col].median()).astype(float)

if 'days_oldest_open_viol' in model_df.columns:
    model_df['days_oldest_open_viol'] = model_df['days_oldest_open_viol'].fillna(0).astype(int)

assert 'zip_code' in model_df.columns, \
    'zip_code missing — required for spatial CV'

n_zips = model_df['zip_code'].nunique(dropna=True)
print(f"model_df: {len(model_df):,} parcels  |  {n_zips} zip codes  |  OVS=1: {model_df['ovs'].mean():.1%}")


In [ ]:
# Replicate 04a's stratified 70/30 split
train_df, test_df = train_test_split(
    model_df, test_size=0.30, random_state=SEED,
    stratify=model_df['ovs'].astype(int).values,
)
train_df = train_df.copy(); test_df = test_df.copy()
print(f"Train: {len(train_df):,}  |  Test: {len(test_df):,}")


## 3. Recipe / spec (mirrors 04a)

`SimpleImputer(median) → VarianceThreshold(0) → RandomForestClassifier(class_weight='balanced')`.
No SMOTE — matches the post-2026-04-28 production pipeline.


In [ ]:
# Reuse RF tune params from 04a if available; otherwise sensible defaults.
rf_tune_csv = PY_PATH / 'rf_tune_results.csv'
if rf_tune_csv.exists():
    s = pd.read_csv(rf_tune_csv)
    BEST_MTRY  = int(s['mtry'].iloc[0])
    BEST_MIN_N = int(s['min_n'].iloc[0])
    print(f"Loaded RF params from 04a tune: mtry={BEST_MTRY} | min_n={BEST_MIN_N}")
else:
    BEST_MTRY  = int(np.floor(np.sqrt(len(model_vars))))
    BEST_MIN_N = 5
    print(f"No saved tune — defaults: mtry={BEST_MTRY} | min_n={BEST_MIN_N}")

def make_rf_pipeline(n_estimators=500, **rf_kwargs):
    return SkPipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('vt',     VarianceThreshold(0.0)),
        ('rf',     RandomForestClassifier(
                        n_estimators=n_estimators,
                        max_features=BEST_MTRY,
                        min_samples_leaf=BEST_MIN_N,
                        class_weight='balanced',
                        random_state=SEED, n_jobs=-1, **rf_kwargs)),
    ])


## 4. Spatial Cross-Validation by ZIP

`GroupKFold` with 10 folds, grouped by `zip_code`. Each fold holds out roughly
10% of zip codes — every parcel from a given ZIP is either entirely in train
or entirely in test, so the model never sees a parcel's neighbors during
training.

> **Sample flag.** `SAMPLE_FOR_CV = True` runs on a 10% stratified sample (~1
> minute). Set to `False` for a final report.


In [ ]:
SAMPLE_FOR_CV = True   # 10% sample — flip to False for final pass

CV_TREES = 200  # CV-only RF, fewer trees for speed; full fit later uses 500.

if SAMPLE_FOR_CV:
    cv_idx = (
        train_df.groupby('zip_code', observed=True, group_keys=False)
                .apply(lambda g: g.sample(frac=0.10, random_state=SEED))
                .index
    )
    cv_data = train_df.loc[cv_idx]
    print(f"Spatial CV on 10% sample: {len(cv_data):,} rows  |  "
          f"{cv_data['zip_code'].nunique()} zip codes")
else:
    cv_data = train_df
    print(f"Spatial CV on full train: {len(cv_data):,} rows  |  "
          f"{cv_data['zip_code'].nunique()} zip codes")


In [ ]:
def cv_zip_metrics(cv_data, n_splits=10, n_estimators=200):
    """Run group-k-fold CV grouped by zip_code; return per-fold metrics dataframe."""
    X = cv_data[model_vars]
    y = cv_data['ovs'].astype(int).values
    groups = cv_data['zip_code'].astype(str).fillna('NA').values

    gkf = GroupKFold(n_splits=n_splits)

    rows = []
    for fold_i, (tr, va) in enumerate(gkf.split(X, y, groups=groups)):
        pipe = make_rf_pipeline(n_estimators=n_estimators)
        pipe.fit(X.iloc[tr], y[tr])
        p = pipe.predict_proba(X.iloc[va])[:, 1]
        # Youden best
        fpr, tpr, _ = roc_curve(y[va], p)
        rows.append({
            'fold':     fold_i + 1,
            'n_train':  len(tr),
            'n_test':   len(va),
            'n_groups_held_out': len(set(groups[va])),
            'roc_auc':  roc_auc_score(y[va], p),
            'pr_auc':   average_precision_score(y[va], p),
            'j_index':  float((tpr - fpr).max()),
        })
    return pd.DataFrame(rows)

print(f"Running spatial CV (10 folds × {CV_TREES}-tree RF) — this takes a few minutes…")
spatial_cv = cv_zip_metrics(cv_data, n_splits=10, n_estimators=CV_TREES)
print('Done.')
spatial_cv.round(4)


In [ ]:
spatial_cv_summary = spatial_cv[['roc_auc', 'pr_auc', 'j_index']].agg(['mean', 'std']).T
spatial_cv_summary['std_err'] = spatial_cv_summary['std'] / np.sqrt(len(spatial_cv))
spatial_cv_summary[['mean', 'std_err']].round(4)


In [ ]:
# Per-fold visual
fig, ax = plt.subplots(figsize=(10, 4.5))
for col, color in [('roc_auc', 'steelblue'), ('j_index', 'tomato')]:
    ax.plot(spatial_cv['fold'], spatial_cv[col], '-o', color=color, label=col, lw=1.5)
    ax.axhline(spatial_cv[col].mean(), ls='--', color=color, alpha=0.4)
ax.set_xlabel('Fold')
ax.set_ylabel('Metric value')
ax.set_title('Spatial CV: Per-Fold Performance by ZIP-Code Group')
ax.legend()
ax.set_xticks(spatial_cv['fold'])
plt.tight_layout()
plt.savefig(GRAPH_PATH / 'spatial_cv_performance.png', dpi=200, bbox_inches='tight')
plt.show()


## 5. LOGO Cross-Validation (Leave-One-ZIP-Out)

`LeaveOneGroupOut`: each ZIP held out once. Hardest generalization test —
simulates deploying to a brand-new neighborhood with no training history.

> **Sample flag.** `SAMPLE_FOR_LOGO = True` uses a random subset of 15 ZIPs
> instead of all ~45. Default ON for ~5-min runtime.


In [ ]:
SAMPLE_FOR_LOGO = True
N_LOGO_ZIPS = 15

logo_data = cv_data  # reuse the spatial-CV sample (already 10% if flag on)

if SAMPLE_FOR_LOGO:
    rng = np.random.default_rng(SEED)
    sampled_zips = rng.choice(logo_data['zip_code'].dropna().unique(),
                              size=min(N_LOGO_ZIPS, logo_data['zip_code'].nunique()),
                              replace=False)
    logo_data = logo_data[logo_data['zip_code'].isin(sampled_zips)]
    print(f"LOGO on {len(sampled_zips)} sampled ZIPs ({len(logo_data):,} parcels)")
else:
    print(f"LOGO on all {logo_data['zip_code'].nunique()} ZIPs ({len(logo_data):,} parcels)")


In [ ]:
def logo_zip_metrics(data, n_estimators=200):
    """LeaveOneGroupOut by zip_code. Returns per-zip metrics."""
    X = data[model_vars]
    y = data['ovs'].astype(int).values
    groups = data['zip_code'].astype(str).fillna('NA').values

    logo = LeaveOneGroupOut()
    rows = []
    n_folds = logo.get_n_splits(groups=groups)
    for fold_i, (tr, va) in enumerate(logo.split(X, y, groups=groups)):
        zip_held = groups[va][0]
        n_pos    = int(y[va].sum())
        if n_pos < 2 or len(va) < 50:
            # Skip ZIPs too small/sparse to compute reliable AUC
            continue
        pipe = make_rf_pipeline(n_estimators=n_estimators)
        pipe.fit(X.iloc[tr], y[tr])
        p = pipe.predict_proba(X.iloc[va])[:, 1]
        rows.append({
            'fold':     fold_i + 1,
            'zip':      zip_held,
            'n':        len(va),
            'n_vacant': n_pos,
            'roc_auc':  roc_auc_score(y[va], p),
            'pr_auc':   average_precision_score(y[va], p) if n_pos > 0 else np.nan,
        })
        if (fold_i + 1) % 5 == 0:
            print(f"  fold {fold_i+1}/{n_folds}: zip={zip_held}, AUC={rows[-1]['roc_auc']:.3f}")
    return pd.DataFrame(rows)

logo_results = logo_zip_metrics(logo_data, n_estimators=CV_TREES)
print(f"\nLOGO complete: {len(logo_results)} ZIPs evaluated")
print(f"  Mean AUC:   {logo_results['roc_auc'].mean():.4f}")
print(f"  Median AUC: {logo_results['roc_auc'].median():.4f}")
print(f"  ZIPs AUC < 0.70: {int((logo_results['roc_auc'] < 0.70).sum())}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, max(6, 0.3 * len(logo_results))))
df_plot = logo_results.sort_values('roc_auc')
colors = ['tomato' if a < 0.70 else 'steelblue' for a in df_plot['roc_auc']]
ax.barh(df_plot['zip'].astype(str), df_plot['roc_auc'], color=colors)
ax.axvline(logo_results['roc_auc'].mean(), ls='--', color='gray',
           label=f"mean AUC = {logo_results['roc_auc'].mean():.3f}")
ax.set_xlim(0, 1)
ax.set_xlabel('ROC-AUC')
ax.set_title('LOGO CV: AUC by Held-Out ZIP\n(red bars = AUC < 0.70; generalization concern)')
ax.legend()
plt.tight_layout()
plt.savefig(GRAPH_PATH / 'logo_cv_by_zip.png', dpi=200, bbox_inches='tight')
plt.show()


## 6. Prediction Confidence Intervals

sklearn's `RandomForestClassifier` exposes the underlying `estimators_` (one
fit per tree). We compute a per-parcel SE as the std of tree-level probability
predictions divided by sqrt(n_trees) — a simpler analog to ranger's
infinitesimal jackknife. The 95% CI is `prob ± 1.96 × SE`.

A parcel with `prob = 0.6` and `ci_width = 0.05` is much more confidently
flagged than one with `prob = 0.6` and `ci_width = 0.30`.


In [ ]:
# Refit the production RF (500 trees) on the full train set so we can pull
# per-tree predictions for SE estimation.
print('Refitting RF (500 trees) for CI estimation…')
rf_full = make_rf_pipeline(n_estimators=500)
rf_full.fit(train_df[model_vars], train_df['ovs'].astype(int).values)
print('Done.')


In [ ]:
def rf_predict_with_ci(pipe, X_df):
    """Predict with per-parcel CIs from the variance across trees.

    Returns a DataFrame with rf_prob, rf_se, ci_lower, ci_upper, ci_width.
    """
    # Feed X through the pre-RF pipeline steps
    X_imputed = pipe.named_steps['impute'].transform(X_df)
    X_vt      = pipe.named_steps['vt'].transform(X_imputed)
    rf_clf    = pipe.named_steps['rf']

    # Each tree predicts a 0/1 vote (or proba if using predict_proba per tree).
    # Use predict_proba on each tree, take prob_class_1.
    tree_preds = np.stack([
        tree.predict_proba(X_vt)[:, 1] for tree in rf_clf.estimators_
    ])  # shape: (n_trees, n_samples)

    rf_prob = tree_preds.mean(axis=0)
    # SE: sample std of tree probs / sqrt(n_trees) — analog to bootstrap SE
    rf_se   = tree_preds.std(axis=0, ddof=1) / np.sqrt(len(rf_clf.estimators_))
    ci_lower = np.maximum(0, rf_prob - 1.96 * rf_se)
    ci_upper = np.minimum(1, rf_prob + 1.96 * rf_se)
    return pd.DataFrame({
        'rf_prob':  rf_prob,
        'rf_se':    rf_se,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'ci_width': ci_upper - ci_lower,
    }, index=X_df.index)

print('Computing CIs on test set…')
ci_df = rf_predict_with_ci(rf_full, test_df[model_vars])
test_with_ci = pd.concat([test_df.reset_index(drop=True),
                          ci_df.reset_index(drop=True)], axis=1)

# Score test parcels with the production ensemble too (logit + RF, calibrated).
# logit raw probs come from the saved logit; rf raw probs come from rf_full
# (this notebook's refit, slightly different from 04a's due to different RF seed
# behavior across spatial fold splits — but the calibration shape holds).
test_with_ci['logit_prob_raw']    = logit_fit.predict_proba(test_df[model_vars])[:, 1]
test_with_ci['rf_prob_raw']       = test_with_ci['rf_prob']
test_with_ci['ensemble_prob_raw'] = 0.5 * test_with_ci['logit_prob_raw'] + 0.5 * test_with_ci['rf_prob_raw']
test_with_ci['ensemble_prob']     = cal_ens.transform(test_with_ci['ensemble_prob_raw'].values)
# Pull the production ensemble_flag (top 1% by raw rank) for the high-uncertainty
# review below — that's what shows up in the city dashboard, not the calibrated
# Youden threshold (which would flag ~14% of parcels).
ens_preds_for_test = preds[['parcel_number', 'ensemble_flag', 'qtile_tier']].copy()
test_with_ci = test_with_ci.merge(
    ens_preds_for_test, on='parcel_number', how='left',
)
print(f"Ensemble scored on test set: AUC = "
      f"{roc_auc_score(test_with_ci['ovs'].astype(int), test_with_ci['ensemble_prob']):.4f}")

summary = pd.DataFrame({
    'metric': ['mean P(vacant)', 'mean SE', 'mean CI width',
               '% CI < 0.10 (high confidence)', '% CI > 0.30 (uncertain)'],
    'value': [
        f"{test_with_ci['rf_prob'].mean():.4f}",
        f"{test_with_ci['rf_se'].mean():.4f}",
        f"{test_with_ci['ci_width'].mean():.4f}",
        f"{(test_with_ci['ci_width'] < 0.10).mean():.1%}",
        f"{(test_with_ci['ci_width'] > 0.30).mean():.1%}",
    ],
})
summary


In [ ]:
# CI width vs predicted probability — where is the model uncertain?
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
binned = test_with_ci.assign(prob_bin=pd.cut(test_with_ci['rf_prob'],
                                             bins=np.arange(0, 1.05, 0.05)))
agg = binned.groupby('prob_bin', observed=True).agg(
    mean_prob=('rf_prob', 'mean'),
    mean_ci=('ci_width', 'mean'),
    n=('rf_prob', 'size'),
).reset_index().dropna()
ax.plot(agg['mean_prob'], agg['mean_ci'], '-', color='steelblue', lw=2)
ax.scatter(agg['mean_prob'], agg['mean_ci'], color='steelblue',
           s=20 + agg['n'].rank(pct=True) * 100, alpha=0.8, zorder=3)
ax.set_xlabel('Predicted P(vacant)')
ax.set_ylabel('Mean CI width')
ax.set_title('CI width vs predicted probability')

ax = axes[1]
for label, color in [(0, 'steelblue'), (1, 'tomato')]:
    sub = test_with_ci.loc[test_with_ci['ovs'] == label, 'ci_width']
    ax.hist(sub, bins=40, alpha=0.5, density=True,
            label=f"OVS = {label}", color=color)
ax.set_xlabel('CI width')
ax.set_ylabel('Density')
ax.set_title('CI width distribution by true OVS')
ax.legend()

plt.tight_layout()
plt.savefig(GRAPH_PATH / 'prediction_ci_analysis.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# High-uncertainty flagged parcels — these are the "we flagged but unsure"
# cases the city should treat with extra care. We use the PRODUCTION flag
# (ensemble_flag = top 1% by raw ensemble rank) — that's what the dashboard
# shows. CIs themselves are still RF-only (only RF has tree variance).
high_unc_mask = (test_with_ci['ci_width'] > 0.30) & (test_with_ci['ensemble_flag'] == 1)
print(f"Parcels in production flag set (top 1% by ensemble): "
      f"{int(test_with_ci['ensemble_flag'].sum()):,}")
print(f"Of those, with CI > 0.30 (high RF uncertainty): "
      f"{int(high_unc_mask.sum()):,}")

high_unc = (
    test_with_ci[high_unc_mask]
    .sort_values('ci_width', ascending=False)
    [['parcel_number', 'ovs', 'ensemble_prob', 'rf_prob', 'rf_se',
      'ci_lower', 'ci_upper', 'ci_width', 'qtile_tier']]
    .head(20)
    .round(4)
)
high_unc


## 7. Sanity Checks

### 7a. Feature importance

In [ ]:
rf_clf = rf_full.named_steps['rf']
vt     = rf_full.named_steps['vt']
surviving = [f for f, k in zip(model_vars, vt.get_support()) if k]
imp_df = pd.DataFrame({
    'feature':    surviving,
    'importance': rf_clf.feature_importances_,
}).sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1], color='steelblue')
ax.set_title('Top 20 features by impurity importance')
plt.tight_layout()
plt.savefig(GRAPH_PATH / 'rf_vip_04b.png', dpi=200, bbox_inches='tight')
plt.show()
imp_df.round(4)


### 7b. Calibration curve

In [ ]:
y_true = test_with_ci['ovs'].astype(int).values

# Calibration for RF (raw) and ensemble (calibrated, what the dashboard shows)
prob_pred_rf,  prob_true_rf  = calibration_curve(
    y_true, test_with_ci['rf_prob'].values, n_bins=10, strategy='quantile')
prob_pred_ens, prob_true_ens = calibration_curve(
    y_true, test_with_ci['ensemble_prob'].values, n_bins=10, strategy='quantile')

fig, ax = plt.subplots(figsize=(8, 6.5))
lim = max(prob_pred_rf.max(), prob_pred_ens.max(),
          prob_true_rf.max(), prob_true_ens.max()) * 1.05
ax.plot([0, lim], [0, lim], '--', color='gray', label='Perfect calibration')
ax.plot(prob_pred_rf,  prob_true_rf,  '-o', color='steelblue', lw=1.5,
        label='RF (raw)', alpha=0.7)
ax.plot(prob_pred_ens, prob_true_ens, '-o', color='black',     lw=2.5,
        label='Vacancy Risk Score (ensemble, calibrated)')
ax.set_xlabel('Mean predicted P(vacant)')
ax.set_ylabel('Observed vacancy rate')
ax.set_title('Calibration: ensemble (production) vs RF alone')
ax.legend()
plt.tight_layout()
plt.savefig(GRAPH_PATH / 'calibration_curve.png', dpi=200, bbox_inches='tight')
plt.show()


### 7c. Known-vacant scoring check

If `cs_truly_active`, `had_vacancy_license`, etc. are present in the test set
(they're excluded from `model_vars` as leaks but still in features_residential),
we can check the model assigns higher scores to those parcels. If it doesn't,
the model is ignoring its strongest non-leakage proxies.


In [ ]:
def safe_col(df, col, default=0):
    return df[col] if col in df.columns else pd.Series(default, index=df.index)

groups = pd.Series('No strong signal', index=test_with_ci.index)
groups[safe_col(test_with_ci, 'cs_truly_active') == 1] = 'Clean & Seal active (strongest)'
groups[(groups == 'No strong signal') & (safe_col(test_with_ci, 'had_vacancy_license') == 1)] = 'Had vacancy license'
groups[(groups == 'No strong signal') & (safe_col(test_with_ci, 'has_open_vacancy_kw') == 1)] = 'Open vacancy-kw violation'

# Sanity scoring uses the PRODUCTION ensemble probability, and flag rate uses
# the production ensemble_flag (top 1% by raw rank), so the table answers
# "how does the dashboard actually treat parcels in each known-signal group?"
sanity = (
    test_with_ci.assign(group=groups)
    .groupby('group', observed=True)
    .agg(n=('ensemble_prob', 'size'),
         ovs1_rate=('ovs', lambda s: float((s == 1).mean())),
         mean_ensemble_prob=('ensemble_prob', 'mean'),
         mean_rf_prob=('rf_prob', 'mean'),
         pct_flagged=('ensemble_flag', lambda s: float(s.mean())))
    .sort_values('mean_ensemble_prob', ascending=False)
    .reset_index()
)
sanity['ovs1_rate']          = sanity['ovs1_rate'].map('{:.1%}'.format)
sanity['pct_flagged']        = sanity['pct_flagged'].map('{:.1%}'.format)
sanity['mean_ensemble_prob'] = sanity['mean_ensemble_prob'].round(4)
sanity['mean_rf_prob']       = sanity['mean_rf_prob'].round(4)
sanity['n']                  = sanity['n'].map('{:,}'.format)
sanity


### 7d. Partial dependence: `n_violations_total`

In [ ]:
# sklearn partial_dependence — works on the full pipeline.
# NB: in older sklearn the result key is 'values' instead of 'grid_values'.
pdp = partial_dependence(
    rf_full, X=train_df[model_vars].sample(min(20000, len(train_df)), random_state=SEED),
    features=['n_violations_total'],
    grid_resolution=30, kind='average', percentiles=(0.01, 0.99),
)
# Handle both old and new sklearn API (values vs grid_values)
xs = pdp.get('grid_values', pdp.get('values'))[0]
ys = pdp['average'][0]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(xs, ys, color='steelblue', lw=2)
ax.set_xlabel('n_violations_total')
ax.set_ylabel('Marginal P(vacant)')
ax.set_title('Partial dependence: P(vacant) vs n_violations_total\n'
             '(other features held at their distribution; expect monotone increase)')
plt.tight_layout()
plt.savefig(GRAPH_PATH / 'partial_dependence_violations.png', dpi=200, bbox_inches='tight')
plt.show()


## 8. Export validation results

In [ ]:
# Export test predictions with CIs — includes ensemble_prob (production score)
# alongside RF stats so downstream consumers don't have to join back.
ci_export_cols = ['parcel_number', 'ovs',
                  'ensemble_prob', 'ensemble_prob_raw', 'ensemble_flag',
                  'rf_prob', 'rf_se', 'ci_lower', 'ci_upper', 'ci_width']
test_with_ci[ci_export_cols].to_csv(OUT_PATH / 'predictions_with_ci.csv', index=False)
print(f"Exported predictions_with_ci.csv: {len(test_with_ci):,} rows")

# Export CV metric tables
spatial_cv.assign(method='spatial_zip').to_csv(OUT_PATH / 'spatial_cv_metrics.csv', index=False)
logo_results.assign(method='logo_zip').to_csv(OUT_PATH / 'logo_cv_metrics.csv', index=False)
print('Exported spatial_cv_metrics.csv, logo_cv_metrics.csv')

# Test-set ensemble metrics — what the dashboard actually deploys.
y_te = test_with_ci['ovs'].astype(int).values
ens_p = test_with_ci['ensemble_prob'].values
ens_test_auc    = roc_auc_score(y_te, ens_p)
ens_test_pr_auc = average_precision_score(y_te, ens_p)

# Validation summary — RF spatial CV is the conservative lower bound; ensemble
# row reports the actual test-set headline number.
summary_tbl = pd.DataFrame({
    'check': [
        'Production model (test set) — ROC-AUC',
        'Production model (test set) — PR-AUC',
        'Spatial CV mean ROC-AUC (RF only, conservative)',
        'Spatial CV mean PR-AUC (RF only, conservative)',
        'LOGO CV mean ROC-AUC (RF only, conservative)',
        'LOGO CV mean PR-AUC (RF only, conservative)',
        'Test mean CI width (RF tree-variance)',
        '% production-flagged parcels with CI > 0.30',
    ],
    'value': [
        f"{ens_test_auc:.4f}",
        f"{ens_test_pr_auc:.4f}",
        f"{spatial_cv['roc_auc'].mean():.4f}",
        f"{spatial_cv['pr_auc'].mean():.4f}",
        f"{logo_results['roc_auc'].mean():.4f}",
        f"{logo_results['pr_auc'].mean():.4f}",
        f"{test_with_ci['ci_width'].mean():.4f}",
        f"{((test_with_ci['ci_width'] > 0.30) & (test_with_ci['ensemble_flag'] == 1)).mean():.2%}",
    ],
})
summary_tbl.to_csv(OUT_PATH / 'validation_summary.csv', index=False)
summary_tbl


---
**Done.** Outputs in `data_py/`. To run a final pass with no sampling, set
`SAMPLE_FOR_CV = False` and `SAMPLE_FOR_LOGO = False` at the top of §4 / §5
(expect ~1–2 hours).


In [ ]:
!jupyter nbconvert --to html /content/drive/MyDrive/PhillyStat_R/PhillyStat360/code/python/04b_model_validation.ipynb